In [3]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

graph_path = "../data/processed/broadway_network.graphml"

G = nx.read_graphml(graph_path)

print(G)

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/broadway_network.graphml'

In [4]:
import os

print(os.getcwd())

/Users/meiayong/Documents/VSCode Projects/mapping-the-broadway-network


In [6]:
graph_path = "data/processed/broadway_network.graphml"

G = nx.read_graphml(graph_path)

print(G)

Graph with 55263 nodes and 1566284 edges


In [8]:
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()

print(f"Nodes: {n_nodes:,}")
print(f"Edges: {n_edges:,}")

density = nx.density(G)

print(f"Density: {density:.8f}")

Nodes: 55,263
Edges: 1,566,284
Density: 0.00102575


In [9]:
degrees = dict(G.degree())

degree_df = (
    pd.DataFrame(
        degrees.items(),
        columns=["performer_id", "degree"]
    )
    .sort_values(
        "degree",
        ascending=False
    )
)

degree_df.head(20)

,performer_id,degree
37048,A_60670,952
42715,A_53589,720
48426,A_68475,701
4940,A_21639,698
939,A_35584,692
50855,A_21617,675
6087,A_52148,652
17582,A_67079,650
24889,A_39487,642
51761,A_66920,640


In [11]:
degree_df.to_csv(
    "data/processed/performer_degree_raw.csv",
    index=False
)

In [14]:
def format_performer_id(df):
    df["performer_id"] = (
        "A_" + df["performer_id"].astype(str)
    )
    return df

In [16]:
cast = pd.read_csv(
    "data/processed/production_cast.csv"
)

cast["performer_id"] = (
    "A_" + cast["performer_id"].astype(str)
)

performer_names = (
    cast[
        ["performer_id", "performer_name"]
    ]
    .drop_duplicates()
)

degree_named = (
    degree_df
    .merge(
        performer_names,
        on="performer_id",
        how="left"
    )
)

degree_named.head(20)

,performer_id,degree,performer_name
0,A_60670,952,George Spelvin
1,A_53589,720,Victor Moore
2,A_68475,701,Le Roi Operti
3,A_21639,698,Dennis King
4,A_35584,692,Dudley Clements
5,A_21617,675,David Wayne
6,A_52148,652,Charles McClelland
7,A_67079,650,Robert Chisholm
8,A_39487,642,Maurice Ellis
9,A_66920,640,Philip Bosco


In [17]:
george = degree_named[
    degree_named["performer_name"]
    .str.contains(
        "George Spelvin",
        case=False,
        na=False
    )
]

george

,performer_id,degree,performer_name
0,A_60670,952,George Spelvin
13018,A_105848,74,"George Spelvin, Jr."
43543,A_382817,17,George Spelvinsky


In [18]:
george_cast = cast[
    cast["performer_id"] == "A_60670"
]

print(
    f"Productions: {george_cast['production_id'].nunique()}"
)

print(
    f"Rows: {len(george_cast)}"
)

Productions: 44
Rows: 44


## Placeholder identity investigation

George Spelvin was identified as the highest-degree node in the raw network.

Although the node represents only 44 productions, it has a degree of 952, suggesting that the identity aggregates multiple anonymous/unidentified performers rather than representing a single individual.

Because the network models individual performer collaboration, placeholder identities were excluded from the cleaned network.

In [20]:
cast_clean = cast[
    cast["performer_id"] != "A_60670"
].copy()

print(cast_clean.shape)

(115633, 4)


In [22]:
cast_clean.to_csv(
    "data/processed/production_cast_clean.csv",
    index=False
)

In [23]:
id_variation = (
    cast_clean
    .groupby("performer_name")["performer_id"]
    .nunique()
    .sort_values(ascending=False)
)

id_variation.head(30)

performer_name
James Young           3
Richard Allen         3
Elizabeth Taylor      3
John Gray             3
James Fox             3
William James         3
John Thomas           3
Keith Davis           3
Charles Brown         3
Robert Thompson       3
Trixie                3
Frank Davis           3
John Kelly            3
Alan Campbell         3
Grace Carroll         3
Michael Williams      2
George Cooper         2
Milton Carney         2
Gene Miller           2
Bob Maxwell           2
Robert B. Williams    2
Albert Hall           2
James Gleason         2
Jack Fletcher         2
Luis Bernardo         2
Mary Young            2
Mary Ellis            2
James Gardner         2
Burton Lewis          2
Frances Williams      2
Name: performer_id, dtype: int64

In [25]:
ambiguous_names = (
    id_variation[id_variation > 1]
    .head(20)
    .index
)

cast_clean[
    cast_clean["performer_name"].isin(ambiguous_names)
].sort_values(
    ["performer_name", "performer_id"]
)[
    [
        "performer_id",
        "performer_name",
        "production_id",
        "character"
    ]
]

,performer_id,performer_name,production_id,character
3316,A_507814,Alan Campbell,10538,Windy
1681,A_67010,Alan Campbell,7885,Mees
4450,A_67010,Alan Campbell,10601,Alan Sands
9233,A_67010,Alan Campbell,10793,Quyen
14777,A_67010,Alan Campbell,11021,James Coleman
...,...,...,...,...
100254,A_73338,Trixie,4643,Wild West Show Dog
26292,A_46815,William James,11665,Ensemble
60970,A_528096,William James,1454,"Gypsy, Gentleman, Bellboy, Waiter"
86035,A_78045,William James,3429,Timmy


In [26]:
cast_clean[
    cast_clean["performer_name"].isna()
]

,performer_id,performer_name,character,production_id


Cleaning complete!

Original cast rows: 115,677
Clean cast rows: 115,632
Rows removed: 45

Original performers: 55,263
Clean performers: 55,261
Performers removed: 2
